In [1]:
#Setup that works on Colab and on a local machine. Run this cell first.
#Local: run `pip install -r requirements.txt` once, and start the notebook from the repo root.
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    #A fresh Colab VM is empty, so get the repo. Safe to re-run: it only clones if the repo isn't there yet.
    if not Path("Data").exists() and not Path("DL_FinalProject").exists():
        subprocess.run(["git", "clone", "https://github.com/mpchouinard/DL_FinalProject.git"], check=True)
    if Path("DL_FinalProject").exists():
        os.chdir("DL_FinalProject")
    #Colab has torch and scikit-learn, but its pandas is too old to read our pickle, and it lacks transformers.
    #Don't install requirements.txt here, it can replace Colab's GPU build of torch.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pandas", "transformers"], check=True)
    if "pandas" in sys.modules:
        print("pandas was already loaded. Restart the session once (Runtime > Restart session), then run again.")

if not Path("Data/c_elegans_preprocessed.pkl").exists():
    raise FileNotFoundError(f"Start the notebook from the repo root. Current folder: {os.getcwd()}")
print("on Colab:", ON_COLAB, "| working directory:", os.getcwd())

on Colab: False | working directory: c:\Repos\DL_FinalProject


## Imports and data

In [ ]:
import gc
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score

from src.embed import load_embeddings

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

with open(Path("Data/c_elegans_preprocessed.pkl"), "rb") as f:
    data = pickle.load(f)
folds = data["folds"]

## Model

In [ ]:
import torch
from torch import nn


class CrossAttentionModel(nn.Module):
    """Predict whether a receptor-peptide pair is an agonist.

    Inputs are cached ESM-2 per-residue embeddings with shape
    (batch, padded_sequence_length, embedding_dim), plus true lengths.
    The 650M embeddings use embedding_dim=1280.
    """

    def __init__(self, embedding_dim=1280, num_heads=8, hidden_dim=256, dropout=0.1):
        super().__init__()
        self.receptor_norm = nn.LayerNorm(embedding_dim)
        self.peptide_norm = nn.LayerNorm(embedding_dim)
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.attended_norm = nn.LayerNorm(embedding_dim)
        self.attended_dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(3 * embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    @staticmethod
    def masked_mean(values, padding_mask):
        # padding_mask is True where a row is padding.
        valid = (~padding_mask).unsqueeze(-1)
        return (values * valid).sum(dim=1) / valid.sum(dim=1).clamp_min(1)

    def forward(self, receptor_residues, peptide_residues, receptor_lengths, peptide_lengths):
        # Cached embeddings are float16; match the model's parameter dtype.
        dtype = self.cross_attention.in_proj_weight.dtype
        receptor_residues = receptor_residues.to(dtype=dtype)
        peptide_residues = peptide_residues.to(dtype=dtype)
        receptor_lengths = torch.as_tensor(receptor_lengths, device=receptor_residues.device)
        peptide_lengths = torch.as_tensor(peptide_lengths, device=peptide_residues.device)

        receptor_padding = (
            torch.arange(receptor_residues.size(1), device=receptor_residues.device)[None, :]
            >= receptor_lengths[:, None]
        )
        peptide_padding = (
            torch.arange(peptide_residues.size(1), device=peptide_residues.device)[None, :]
            >= peptide_lengths[:, None]
        )

        receptor = self.receptor_norm(receptor_residues)
        peptide = self.peptide_norm(peptide_residues)

        # Each peptide residue queries the receptor residues. Padded receptor
        # positions cannot receive attention. Padded peptide positions are
        # excluded when we summarize the attended sequence below.
        attended, _ = self.cross_attention(
            query=peptide,
            key=receptor,
            value=receptor,
            key_padding_mask=receptor_padding,
            need_weights=False,
        )
        attended = self.attended_norm(peptide + self.attended_dropout(attended))

        pair_features = torch.cat(
            [
                self.masked_mean(peptide, peptide_padding),
                self.masked_mean(receptor, receptor_padding),
                self.masked_mean(attended, peptide_padding),
            ],
            dim=-1,
        )
        # One raw logit per pair; the training code will choose the loss.
        return self.classifier(pair_features).squeeze(-1)

## Encoder
Turns names into per-residue embeddings.

In [ ]:
class CachedEncoder:
    """Looks up precomputed (frozen) ESM-2 embeddings by name."""

    def __init__(self, tag="650M", device=DEVICE, folder="embeddings"):
        self.device = device
        self.tables = {}
        for kind, file_prefix in [("receptor", "receptors"), ("peptide", "peptides")]:
            loaded = load_embeddings(f"{folder}/{file_prefix}_{tag}.npz")
            self.tables[kind] = {
                "row": {name: i for i, name in enumerate(loaded["names"])},
                "lengths": torch.as_tensor(loaded["lengths"], dtype=torch.long, device=device),
                "residues": torch.as_tensor(loaded["residues"], device=device),
            }

    def __call__(self, kind, names):
        table = self.tables[kind]
        rows = torch.as_tensor([table["row"][name] for name in names], device=self.device)
        lengths = table["lengths"][rows]
        longest = int(lengths.max())  # drop padding columns this batch doesn't need
        return table["residues"][rows, :longest], lengths


encoder = CachedEncoder("650M")

## Batching

In [ ]:
def iterate_batches(pairs, batch_size, shuffle, generator=None):
    receptors = pairs["GPCR name"].to_numpy()
    peptides = pairs["Peptide"].to_numpy()
    labels = pairs["binds"].to_numpy().astype(np.float32)

    if shuffle:
        order = torch.randperm(len(pairs), generator=generator).numpy()
    else:
        order = np.arange(len(pairs))

    for start in range(0, len(pairs), batch_size):
        batch = order[start:start + batch_size]
        yield list(receptors[batch]), list(peptides[batch]), torch.as_tensor(labels[batch])


def forward_batch(model, encoder, receptor_names, peptide_names):
    receptor_residues, receptor_lengths = encoder("receptor", receptor_names)
    peptide_residues, peptide_lengths = encoder("peptide", peptide_names)
    return model(receptor_residues, peptide_residues, receptor_lengths, peptide_lengths)

## Losses

In [ ]:
def bce_loss(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets)


def focal_loss(logits, targets, gamma=2.0, alpha=0.25):
    #BCE that down-weights easy pairs
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p_true = torch.exp(-bce)
    class_weight = alpha * targets + (1 - alpha) * (1 - targets)
    return (class_weight * (1 - p_true) ** gamma * bce).mean()

## Evaluation
The main metric is per-receptor average precision, averaged over receptors (mAP), as in the paper.

In [ ]:
def per_receptor_ap(pairs, scores):
    #average precision for each receptor that has at least one agonist
    scored = pairs.assign(score=scores)
    ap = {}
    for receptor, group in scored.groupby("GPCR name"):
        if group["binds"].any():
            ap[receptor] = average_precision_score(group["binds"], group["score"])
    return ap


@torch.no_grad()
def evaluate(model, encoder, pairs, batch_size=256):
    model.eval()
    scores = []
    for receptor_names, peptide_names, _ in iterate_batches(pairs, batch_size, shuffle=False):
        scores.append(forward_batch(model, encoder, receptor_names, peptide_names).float().cpu())
    scores = torch.cat(scores).numpy()

    labels = pairs["binds"].to_numpy().astype(int)
    receptor_ap = per_receptor_ap(pairs, scores)
    agonist_fraction = pairs.groupby("GPCR name")["binds"].mean()
    return {
        "map": np.mean(list(receptor_ap.values())),                       # mean of per-receptor AP
        "random_map": agonist_fraction[list(receptor_ap)].mean(),         # what a random ranking would get
        "receptor_ap": receptor_ap,
        "auprc": average_precision_score(labels, scores),                 # all pairs pooled
        "auroc": roc_auc_score(labels, scores),
        "n_agonists": int(labels.sum()),
    }

## Training loop

In [ ]:
def sample_epoch_pairs(train_pairs, negatives_per_agonist, seed):
    #all agonists plus a fresh random sample of negatives
    agonists = train_pairs[train_pairs["binds"]]
    negatives = train_pairs[~train_pairs["binds"]]
    n_negatives = min(len(negatives), negatives_per_agonist * len(agonists))
    return pd.concat([agonists, negatives.sample(n_negatives, random_state=seed)])


def train_one_fold(fold, config, encoder, seed=0):
    """Trains on the fold's training pairs and keeps the epoch with the best validation score.
    The test set is not used here. Returns the model, its saved weights and its validation scores."""

    torch.manual_seed(seed)
    generator = torch.Generator().manual_seed(seed)

    model = config["model_factory"]().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])

    best_score, best_epoch, best_weights, best_val = -1.0, -1, None, None
    for epoch in range(config["epochs"]):
        epoch_pairs = sample_epoch_pairs(fold["train"], config["negatives_per_agonist"], seed + epoch)

        model.train()
        total_loss, n_batches = 0.0, 0
        for receptor_names, peptide_names, labels in iterate_batches(
                epoch_pairs, config["batch_size"], shuffle=True, generator=generator):
            optimizer.zero_grad()
            logits = forward_batch(model, encoder, receptor_names, peptide_names)
            loss = config["loss_fn"](logits, labels.to(DEVICE))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1

        val = evaluate(model, encoder, fold["hyperparameter_tuning"])
        print(f"  epoch {epoch:2d}  train loss {total_loss / n_batches:.4f}  "
              f"val mAP {val['map']:.4f}  val AUPRC {val['auprc']:.4f}")

        if val[config["select_by"]] > best_score:
            best_score, best_epoch, best_val = val[config["select_by"]], epoch, val
            best_weights = {name: w.detach().cpu().clone() for name, w in model.state_dict().items()}
        elif epoch - best_epoch >= config["patience"]:
            print(f"  early stop (best epoch {best_epoch})")
            break

    model.load_state_dict(best_weights)
    return model, best_weights, best_val


def run_all_folds(folds, config, encoder, run_name, seed=0):
    """Trains every fold, scores its test set once, and saves each fold's weights to checkpoints/run_name/."""

    checkpoint_folder = Path("checkpoints") / run_name
    checkpoint_folder.mkdir(parents=True, exist_ok=True)

    rows = []
    for fold_id, fold in enumerate(folds):
        print(f"fold {fold_id}: test families {fold['test_families']}")
        model, weights, val = train_one_fold(fold, config, encoder, seed=seed)
        test = evaluate(model, encoder, fold["test"])
        print(f"  -> test mAP {test['map']:.4f} (random {test['random_map']:.4f})  "
              f"AUPRC {test['auprc']:.4f}  AUROC {test['auroc']:.4f}")

        rows.append({"fold": fold_id, "val_map": val["map"], "val_auprc": val["auprc"],
                     "test_map": test["map"], "test_random_map": test["random_map"],
                     "test_auprc": test["auprc"], "test_auroc": test["auroc"],
                     "test_agonists": test["n_agonists"], "receptor_ap": test["receptor_ap"]})
        torch.save(weights, checkpoint_folder / f"fold{fold_id}.pt")

        del model, weights  # free this fold's GPU memory before the next fold
        gc.collect()
        torch.cuda.empty_cache()
    return pd.DataFrame(rows)

## Parameter sweep

In [ ]:
#Random search. Each trial trains on the four folds with the most validation agonists and records
#the average best validation mAP. The test sets are not used.
sweep_rows = []
for trial in range(15):
    lr = random.choice([3e-5, 1e-4, 3e-4])
    dropout = random.choice([0.1, 0.3, 0.5])
    weight_decay = random.choice([0.01, 0.1])
    negatives_per_agonist = random.choice([4, 10])
    loss_name = random.choice(["bce", "focal"])
    print(f"trial {trial}: lr {lr}, dropout {dropout}, weight decay {weight_decay}, "
          f"negatives {negatives_per_agonist}, loss {loss_name}")

    config = {
        "model_factory": lambda: CrossAttentionModel(embedding_dim=1280, dropout=dropout),
        "negatives_per_agonist": negatives_per_agonist,
        "loss_fn": bce_loss if loss_name == "bce" else focal_loss,
        "lr": lr,
        "weight_decay": weight_decay,
        "batch_size": 32,
        "epochs": 30,
        "patience": 5,
        "select_by": "map",
        "grad_clip": 1.0,
    }

    val_maps = []
    for fold_id in [5, 3, 8, 4]:
        model, weights, val = train_one_fold(folds[fold_id], config, encoder)
        val_maps.append(val["map"])
        del model, weights
        gc.collect()
        torch.cuda.empty_cache()

    sweep_rows.append({"lr": lr, "dropout": dropout, "weight_decay": weight_decay,
                       "negatives_per_agonist": negatives_per_agonist, "loss": loss_name,
                       "val_map": np.mean(val_maps)})
    pd.DataFrame(sweep_rows).to_csv("sweep_results.csv", index=False)  # saved after every trial
    print(f"  -> mean val mAP {np.mean(val_maps):.4f}")

print(pd.DataFrame(sweep_rows).sort_values("val_map", ascending=False).to_string(index=False))

## Final run

In [ ]:
config = {
    "model_factory": lambda: CrossAttentionModel(embedding_dim=1280, dropout=0.1),
    "negatives_per_agonist": 4,
    "loss_fn": bce_loss,        # or focal_loss
    "lr": 1e-4,
    "weight_decay": 0.01,
    "batch_size": 32,
    "epochs": 30,
    "patience": 5,
    "select_by": "map",         # validation score that picks the best epoch: "map" or "auprc"
    "grad_clip": 1.0,
}

results = run_all_folds(folds, config, encoder, run_name="cross_attention_bce")
print(results.drop(columns="receptor_ap").to_string(index=False))
print(results[["test_map", "test_random_map", "test_auprc", "test_auroc"]].agg(["mean", "std"]))